# 12 — NER
**Goal:** Identify and classify named entities (people, companies, skills).

## 1. Built-in spaCy NER

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")
text = "Srivatsa Gorti worked at Google in Mountain View, CA from 2020 to 2023."
for e in nlp(text).ents:
    print(f"  {e.text:25s} {e.label_:10s} {spacy.explain(e.label_)}")

## 2. The Problem: Skills Not Detected

In [ ]:
text = "Expert in Python, TensorFlow, and AWS at Microsoft since 2020."
doc = nlp(text)
for e in doc.ents:
    print(f"  {e.text:20s} {e.label_}")
print("\nNotice: Python, TensorFlow, AWS are NOT found - they need custom NER!")

## 3. Adding Custom Skills with EntityRuler

In [ ]:
from spacy.pipeline import EntityRuler
nlp2 = spacy.load("en_core_web_sm")
ruler = nlp2.add_pipe("entity_ruler", before="ner")
ruler.add_patterns([
    {"label": "SKILL", "pattern": "Python"},
    {"label": "SKILL", "pattern": "TensorFlow"},
    {"label": "SKILL", "pattern": "AWS"},
    {"label": "SKILL", "pattern": [{"LOWER": "machine"}, {"LOWER": "learning"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "deep"}, {"LOWER": "learning"}]},
])
doc2 = nlp2(text)
for e in doc2.ents:
    print(f"  {e.text:20s} {e.label_}")

## 4. Hybrid Regex + NER Extractor

In [ ]:
import re
class ResumeNER:
    def __init__(self):
        self.nlp = spacy.load("en_core_web_sm")
        self.known_skills = {"Python","TensorFlow","PyTorch","AWS","Docker","Kubernetes","SQL"}
    def extract(self, text):
        doc = self.nlp(text)
        result = {"people": set(), "orgs": set(), "skills": set()}
        for e in doc.ents:
            if e.label_ == "PERSON": result["people"].add(e.text)
            elif e.label_ == "ORG": result["orgs"].add(e.text)
        for s in self.known_skills:
            if re.search(r"\\b" + re.escape(s) + r"\\b", text, re.IGNORECASE):
                result["skills"].add(s)
        return result

ner = ResumeNER()
print(ner.extract("Srivatsa knows Python, AWS, and Docker. He worked at Google."))